In [27]:
from pyspark.sql import SparkSession
from minio import Minio

spark = SparkSession.builder.appName("nyc-taxi-parquet").getOrCreate()

# Spark uses Hadoop configs internally to connect to storage systems like S3 / MinIO
hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "minioadmin")
hadoop_conf.set("fs.s3a.secret.key", "minioadmin123")
hadoop_conf.set("fs.s3a.path.style.access", "true")        # Required for MinIO because it uses path-style URLs instead of virtual-hosted style
                                                           # Example path-style: http://minio:9000/bucket-name/object
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")  # MinIO is running on HTTP locally (not HTTPS)

In [28]:
# ---------- Connect to MinIO ----------

minio_client = Minio(
    "minio:9000",
    access_key="minioadmin",
    secret_key="minioadmin123",
    secure=False
)

bucket_name = "nyc-taxi"

# Print all objects present inside the bucket
# This helps verify whether parquet files exist and are accessible
print("\n" + "="*60)
print(f"Objects inside '{bucket_name}':")
for obj in minio_client.list_objects(bucket_name, recursive=True):
    print(f" - {obj.object_name} ({obj.size} bytes)")
print("="*60)
print("NYC Taxi Parquet Upload Complete")
print("="*60)


Objects inside 'nyc-taxi':
 - 2023/yellow_tripdata_2023-01.parquet (47673370 bytes)
 - 2023/yellow_tripdata_2023-02.parquet (47748012 bytes)
 - 2023/yellow_tripdata_2023-03.parquet (56127762 bytes)
 - 2023/yellow_tripdata_2023-04.parquet (54222699 bytes)
 - 2023/yellow_tripdata_2023-05.parquet (58654627 bytes)
 - 2023/yellow_tripdata_2023-06.parquet (54999465 bytes)
 - 2023/yellow_tripdata_2023-07.parquet (48361828 bytes)
 - 2023/yellow_tripdata_2023-08.parquet (48152353 bytes)
 - 2023/yellow_tripdata_2023-09.parquet (47895515 bytes)
 - 2023/yellow_tripdata_2023-10.parquet (59009059 bytes)
 - 2023/yellow_tripdata_2023-11.parquet (56094653 bytes)
 - 2023/yellow_tripdata_2023-12.parquet (56804275 bytes)
 - 2024/yellow_tripdata_2024-01.parquet (49961641 bytes)
 - 2024/yellow_tripdata_2024-02.parquet (50349284 bytes)
 - 2024/yellow_tripdata_2024-03.parquet (60078280 bytes)
 - 2024/yellow_tripdata_2024-04.parquet (59133625 bytes)
 - 2024/yellow_tripdata_2024-05.parquet (62553128 bytes)
 - 

In [29]:
from pyspark.sql.functions import col

# Month 1 (January) has a different schema than the other files. Therefore, some of its columns' data types have been changed.
# Also, the Airport_fee column name was different, so it has been changed as well.

# Read January separately and cast to match the standard schema
df_jan = spark.read.parquet("s3a://nyc-taxi/2023/yellow_tripdata_2023-01.parquet")

df_jan = df_jan \
    .withColumn("VendorID", col("VendorID").cast("integer")) \
    .withColumn("passenger_count", col("passenger_count").cast("long")) \
    .withColumn("RatecodeID", col("RatecodeID").cast("long")) \
    .withColumn("PULocationID", col("PULocationID").cast("integer")) \
    .withColumn("DOLocationID", col("DOLocationID").cast("integer")) \
    .withColumnRenamed("airport_fee", "Airport_fee")

# Read the rest (Feb-Dec)
df_rest = spark.read.parquet(
    "s3a://nyc-taxi/2023/yellow_tripdata_2023-0[2-9].parquet",
    "s3a://nyc-taxi/2023/yellow_tripdata_2023-1[0-2].parquet"
)

# Union them together
df1 = df_jan.union(df_rest)

In [41]:
# Combining parquet files from all the folders to create a single dataframe.

from pyspark.sql.functions import lit

df2 = spark.read.parquet("s3a://nyc-taxi/2024/*.parquet")
df_1_2 = df1.union(df2)

# Newer congestion charge (column name - cbd_congestion_fee) was introduced in 2025's datasets. 
# So, to union it with previous years' dataframes, we created the same column in previous years' dataframes.

df_1_2 = df_1_2.withColumn('cbd_congestion_fee', lit(0.0))
df3 = spark.read.parquet("s3a://nyc-taxi/2025/*.parquet")
df = df_1_2.union(df3)
df.count()

111036384

# EDA

In [31]:
df.describe().show()

+-------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+-----------------+--------------------+-------------------+-------------------+
|summary|          VendorID|   passenger_count|    trip_distance|        RatecodeID|store_and_fwd_flag|      PULocationID|      DOLocationID|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|     total_amount|congestion_surcharge|        Airport_fee| cbd_congestion_fee|
+-------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------

We can see three problems with the data through the summary statistics - 
1. Many rows don't contain any value in the passenger count column.
2. The min values of fees and amount-related columns are negative.
3. The max values for the trip_distance and fare_amount columns are too high.
4. Zero trip_distance and fare_amount values.

### Going through the rows with null values in the passenger_count column.

In [32]:
from pyspark.sql.functions import col

df_null_passenger = df.filter(col("passenger_count").isNull())
df_null_passenger.describe().show()

+-------+------------------+---------------+------------------+----------+------------------+------------------+------------------+------------+------------------+------------------+------------------+------------------+------------------+---------------------+------------------+--------------------+-----------+------------------+
|summary|          VendorID|passenger_count|     trip_distance|RatecodeID|store_and_fwd_flag|      PULocationID|      DOLocationID|payment_type|       fare_amount|             extra|           mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+-------+------------------+---------------+------------------+----------+------------------+------------------+------------------+------------+------------------+------------------+------------------+------------------+------------------+---------------------+------------------+--------------------+-----------+------------------+
|

In [33]:
df_null_passenger.groupBy("VendorID").count().show()

+--------+--------+
|VendorID|   count|
+--------+--------+
|       1| 2391564|
|       2|10333323|
|       6|   19291|
+--------+--------+



In [34]:
df_null_passenger.groupBy("PULocationID").count().show()

+------------+------+
|PULocationID| count|
+------------+------+
|         148|246081|
|         243| 30060|
|          31|   923|
|         137|180374|
|          85| 13129|
|          65| 43493|
|         255| 84605|
|          53|  2760|
|         133|  8166|
|          78|  6104|
|         155|  6324|
|         108|  2026|
|         211|137590|
|         193| 15606|
|          34|  6386|
|         101|  2033|
|         126|  5921|
|          81|  3948|
|          28|  9754|
|         210|  4470|
+------------+------+
only showing top 20 rows



In [35]:
df_null_passenger.groupBy("DOLocationID").count().show()

+------------+------+
|DOLocationID| count|
+------------+------+
|         148|236461|
|         243| 85820|
|          31|  1774|
|         137|179076|
|          85| 13306|
|          65| 22668|
|         255| 41330|
|          53|  6211|
|         133|  7503|
|          78|  7578|
|         155|  7884|
|         108|  3590|
|         211|125434|
|         193| 11754|
|          34|  4322|
|         126|  6356|
|         101|  2569|
|          81|  4304|
|          28|  9795|
|         210|  6891|
+------------+------+
only showing top 20 rows



This shows there is not a pattern with null values for a specific vendor or location. Therefore we can go ahead with removing these rows with null values.

In [42]:
df = df.dropna()
df.count()

### For tipping prediction/analysis, negative amounts are not normal “real trips”, so we are going to exclude them

In [44]:
df = df.filter(
    (col("fare_amount") >= 0) &
    (col("trip_distance") >= 0) &
    (col("total_amount") >= 0) &
    (col("tip_amount") >= 0) &
    (col("extra") >= 0) &
    (col("mta_tax") >= 0) &
    (col("tolls_amount") >= 0) &
    (col("improvement_surcharge") >= 0) &
    (col("congestion_surcharge") >= 0) &
    (col("Airport_fee") >= 0) &
    (col("cbd_congestion_fee") >= 0)
)

df.describe().show()

+-------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+------------------+-------------------+
|summary|          VendorID|   passenger_count|     trip_distance|        RatecodeID|store_and_fwd_flag|      PULocationID|     DOLocationID|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|       Airport_fee| cbd_congestion_fee|
+-------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------

### Now, we will do the outlier analysis for trip distance

In [60]:
df.select("trip_distance").describe().show()

+-------+------------------+
|summary|     trip_distance|
+-------+------------------+
|  count|          96746457|
|   mean|3.5408033883862964|
| stddev| 76.14819849332225|
|    min|               0.0|
|    max|          161726.1|
+-------+------------------+



In [61]:
q1, median, q3 = df.approxQuantile("trip_distance", [0.25, 0.5, 0.75], 0.01)

iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("Q1:", q1)
print("Median:", median)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower Bound:", lower)
print("Upper Bound:", upper)

Q1: 1.02
Median: 1.72
Q3: 3.28
IQR: 2.26
Lower Bound: -2.3699999999999997
Upper Bound: 6.67


In [62]:
from pyspark.sql.functions import col

outliers_df = df.filter((col("trip_distance") < lower) | (col("trip_distance") > upper))
normal_df   = df.filter((col("trip_distance") >= lower) & (col("trip_distance") <= upper))

print("Normal Rows:", normal_df.count())
print("Outlier Rows:", outliers_df.count())

Normal Rows: 83708873
Outlier Rows: 13037584


The IQR method won't be applicable here since the outlier rows are in millions. So, we will take 100 as the outlier distance.

In [66]:
long_distance_df = df.filter(col("trip_distance") > 100)

print("Long Distance Rows:", long_distance_df.count())

High Distance Rows: 2012


In [67]:
total_rows = df.count()
long_distance_rows = long_distance_df.count()

print("Long Distance Rows %:", (long_distance_rows / total_rows) * 100)

High Distance Rows %: 0.0020796627208787607


100 seems like a good value to take as an outlier. Therefore, we will filter out the rows that contain a trip distance greater than 100.

In [69]:
df = df.filter((col("trip_distance") <= 100))

print("Rows after trimming:", df.count())

Rows after trimming: 96744445


In [70]:
df.describe().show()

+-------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+-------------------+
|summary|          VendorID|   passenger_count|    trip_distance|        RatecodeID|store_and_fwd_flag|      PULocationID|      DOLocationID|      payment_type|       fare_amount|             extra|           mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee| cbd_congestion_fee|
+-------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+-------

We can still see very high total amount values. Next, we will filter for that (i.e., amount greater than 500).  

In [73]:
high_cost_df = df.filter(col("total_amount") > 500)
print("High Cost Rows:", high_cost_df.count())

High Cost Rows: 1761


In [74]:
total_rows = df.count()
high_cost_df = high_cost_df.count()

print("High Cost Rows %:", (high_cost_df / total_rows) * 100)

High Cost Rows %: 0.0018202595508196879


In [75]:
df = df.filter((col("total_amount") <= 500))

print("Rows after trimming:", df.count())

Rows after trimming: 96742684


In [76]:
df.describe().show()

+-------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+--------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+-------------------+
|summary|          VendorID|   passenger_count|    trip_distance|        RatecodeID|store_and_fwd_flag|      PULocationID|      DOLocationID|      payment_type|      fare_amount|             extra|             mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee| cbd_congestion_fee|
+-------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+--------------------+------------------+------------------+---------------------+----

### Check for zero distance and zero fare rows.

In [81]:
zero_distance_df = df.filter(col("trip_distance") == 0)
print("Zero Distance Rows:", zero_distance_df.count())

Zero Distance Rows: 1179794


In [82]:
zero_fare_df = df.filter(col("fare_amount") == 0)
print("Zero Fare Rows:", zero_fare_df.count())

Zero Fare Rows: 30019


In [86]:
zero_distance_df.select("fare_amount", "total_amount", "tip_amount", "payment_type") \
  .describe() \
  .show()

+-------+------------------+-----------------+-----------------+------------------+
|summary|       fare_amount|     total_amount|       tip_amount|      payment_type|
+-------+------------------+-----------------+-----------------+------------------+
|  count|           1179794|          1179794|          1179794|           1179794|
|   mean|30.403765801486582|37.93141550134817|3.454934920842125|  1.67966950162486|
| stddev| 39.06620980276862|42.90768346177385|7.665664297335325|0.9476936000172407|
|    min|               0.0|              0.0|              0.0|                 1|
|    max|             500.0|            500.0|            494.0|                 5|
+-------+------------------+-----------------+-----------------+------------------+



In [88]:
zero_fare_df.select("trip_distance", "fare_amount", "total_amount", "tip_amount", "payment_type") \
  .describe() \
  .show()

+-------+------------------+-----------+-----------------+------------------+------------------+
|summary|     trip_distance|fare_amount|     total_amount|        tip_amount|      payment_type|
+-------+------------------+-----------+-----------------+------------------+------------------+
|  count|             30019|      30019|            30019|             30019|             30019|
|   mean|1.9949012292214916|        0.0|2.604796295679404|0.7418185149405377|2.5519171191578667|
| stddev|  5.50628341973777|        0.0|9.404846416926187| 8.269101898021075|1.0091499106679318|
|    min|               0.0|        0.0|              0.0|               0.0|                 1|
|    max|              93.5|        0.0|           286.43|             270.0|                 5|
+-------+------------------+-----------+-----------------+------------------+------------------+



Because trip_distance and fare_amount are important features, and 0-distance or 0-fare-amount rows can confuse our analysis. We will filter them out.

In [89]:
df = df.filter((col("trip_distance") > 0) & (col("fare_amount") > 0))
df.count()

95547339

In [90]:
df.describe().show()

+-------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+-------------------+
|summary|          VendorID|   passenger_count|     trip_distance|       RatecodeID|store_and_fwd_flag|      PULocationID|      DOLocationID|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee| cbd_congestion_fee|
+-------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+----

### The dataset is clean and ready to be used for modelling.